In [1]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
import optuna
import pacmap
import seaborn as sns
import matplotlib.pyplot as plt
import joblib

from pyDRMetrics.pyDRMetrics import DRMetrics

from sklearn.mixture import GaussianMixture
from sklearn.metrics import (
    silhouette_score,
    calinski_harabasz_score,
    adjusted_rand_score
)

optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings("ignore", category=FutureWarning)


# ============================================================
# 1. Diretórios
# ============================================================

# Se o script estiver dentro da pasta "notebooks" ou "scripts", usamos Path.cwd().parent ou Path.cwd()
PROJECT_DIR = Path.cwd().parent if Path.cwd().name in ["notebooks", "scripts"] else Path.cwd()

DATA_DIR = PROJECT_DIR / "data" / "processed"
RESULTS_DIR = PROJECT_DIR / "results"

OPTUNA_DIR = RESULTS_DIR / "optuna"
EMBEDDINGS_DIR = RESULTS_DIR / "embeddings"
METRICS_DIR = RESULTS_DIR / "metrics"
FIGURES_DIR = RESULTS_DIR / "figures"

for directory in [OPTUNA_DIR, EMBEDDINGS_DIR, METRICS_DIR, FIGURES_DIR]:
    directory.mkdir(parents=True, exist_ok=True)



d:\Projetos Python\Mestrado\RBras\.venv-1\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Carregando os dados

In [4]:
csv_path = DATA_DIR / "df_pca_50_mad5000.csv"

if not csv_path.exists():
    raise FileNotFoundError(f"Arquivo não encontrado em: {csv_path}. Execute o script de pré-processamento primeiro.")

print(f"Carregando a base de 50 componentes principais em CSV: {csv_path.name}")
df_pca = pd.read_csv(csv_path, index_col=0)

# Carregamento dos dados clínicos (Subtipos PAM50)
clinical_final = pd.read_csv(
    DATA_DIR / "clinical_final.csv",
    index_col=0
)

# Mantém apenas a coluna de subtipo
clinical_final = clinical_final[["SUBTYPE"]]

# Alinha os dados clínicos mantendo estritamente as amostras aprovadas no QC (presentes em df_pca)
clinical_final = clinical_final.loc[df_pca.index]

# Verificações
assert df_pca.index.equals(clinical_final.index), (
    "Os índices da matriz PCA e da tabela clínica não coincidem."
)

assert not df_pca.isnull().values.any(), (
    "A matriz PCA contém valores ausentes."
)

assert not clinical_final["SUBTYPE"].isnull().any(), (
    "A tabela clínica contém subtipos ausentes."
)

X_data = df_pca.values  # Matriz de 50 PCs que servirá de alta dimensão
y_true = clinical_final["SUBTYPE"].values

print(f"\nNúmero de amostras aprovadas no QC : {X_data.shape[0]}")
print(f"Número de Componentes Principais (PCA) : {X_data.shape[1]}")
print(f"Número de subtipos PAM50           : {clinical_final['SUBTYPE'].nunique()}")

print("\nDistribuição dos subtipos:")
print(clinical_final["SUBTYPE"].value_counts())

Carregando a base de 50 componentes principais em CSV: df_pca_50_mad5000.csv

Número de amostras aprovadas no QC : 964
Número de Componentes Principais (PCA) : 50
Número de subtipos PAM50           : 5

Distribuição dos subtipos:
SUBTYPE
BRCA_LumA      491
BRCA_LumB      192
BRCA_Basal     170
BRCA_Her2       76
BRCA_Normal     35
Name: count, dtype: int64


## Função Objetivo PaCMAP


In [7]:
def executar_pacmap(X, params):
    """
    Executa o PaCMAP utilizando os hiperparâmetros informados.
    """

    reducer = pacmap.PaCMAP(
        n_components=params["n_components"],
        n_neighbors=params["n_neighbors"],
        MN_ratio=params["MN_ratio"],
        FP_ratio=params["FP_ratio"],
        random_state=42
    )

    Z = reducer.fit_transform(X, init="pca")

    return Z


def calcular_metricas_reducao(X, Z):
    """
    Calcula as métricas de preservação da estrutura dos dados.
    """

    dr_metrics = DRMetrics(X, Z)

    q_local = dr_metrics.Qlocal
    auc_qnn = dr_metrics.AUC

    return q_local, auc_qnn


def calcular_metricas_clustering(Z, labels):
    """
    Calcula Silhouette e Calinski-Harabasz.
    """

    n_clusters = len(np.unique(labels))

    if n_clusters < 2 or n_clusters >= len(labels):
        raise ValueError("Número inválido de clusters para avaliação.")

    sil_score = silhouette_score(Z, labels)

    ch_score = calinski_harabasz_score(Z, labels)

    # Mantém a normalização utilizada no código original
    ch_score_norm = ch_score / (ch_score + 1000)

    return sil_score, ch_score, ch_score_norm


def calcular_score_reducao(X, Z):
    """
    Score utilizado para avaliar a redução de dimensionalidade.
    """

    q_local, auc_qnn = calcular_metricas_reducao(X, Z)

    score = 0.5 * q_local + 0.5 * auc_qnn

    return score, q_local, auc_qnn


def calcular_score_clustering(Z, labels):
    """
    Score utilizado para avaliar o clustering.
    """

    sil_score, ch_score, ch_score_norm = (
        calcular_metricas_clustering(Z, labels)
    )

    score = 0.5 * sil_score + 0.5 * ch_score_norm

    return score, sil_score, ch_score, ch_score_norm

In [10]:
def objective_pacmap(trial):

    params = {
        "n_neighbors": trial.suggest_int(
            "n_neighbors", 10, 125
        ),

        "MN_ratio": trial.suggest_float(
            "MN_ratio", 0.1, 3.0
        ),

        "FP_ratio": trial.suggest_float(
            "FP_ratio", 0.1, 3.0
        ),

        "n_components": trial.suggest_int(
            "n_components", 2, 10
        )
    }

    Z = executar_pacmap(X_data, params)

    try:
        score_dr, q_local, auc_qnn = calcular_score_reducao(
            X_data, Z
        )

    except Exception as error:
        print(f"Erro nas métricas de redução: {error}")
        raise optuna.TrialPruned()

    trial.set_user_attr("Qlocal", float(q_local))
    trial.set_user_attr("AUC_QNN", float(auc_qnn))

    return score_dr

## Execução do Optuna PaCMAP

In [13]:
study_pacmap = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42),
    study_name="PaCMAP_VST_MAD_5000",
    storage=f"sqlite:///{OPTUNA_DIR / 'pacmap_vst_mad_5000.db'}",
    load_if_exists=True
)

study_pacmap.optimize(
    objective_pacmap,
    n_trials=200
)

print("Melhor score:", study_pacmap.best_value)

print("\nMelhores hiperparâmetros:")
print(study_pacmap.best_params)

Note: `n_components != 2` have not been thoroughly tested.
Note: `n_components != 2` have not been thoroughly tested.
Note: `n_components != 2` have not been thoroughly tested.
Note: `n_components != 2` have not been thoroughly tested.
Note: `n_components != 2` have not been thoroughly tested.
Note: `n_components != 2` have not been thoroughly tested.
Note: `n_components != 2` have not been thoroughly tested.
Note: `n_components != 2` have not been thoroughly tested.
Note: `n_components != 2` have not been thoroughly tested.
Note: `n_components != 2` have not been thoroughly tested.
Note: `n_components != 2` have not been thoroughly tested.
Note: `n_components != 2` have not been thoroughly tested.
Note: `n_components != 2` have not been thoroughly tested.
Note: `n_components != 2` have not been thoroughly tested.
Note: `n_components != 2` have not been thoroughly tested.
Note: `n_components != 2` have not been thoroughly tested.
Note: `n_components != 2` have not been thoroughly teste

Melhor score: 0.6355449744260157

Melhores hiperparâmetros:
{'n_neighbors': 22, 'MN_ratio': 2.689803035941912, 'FP_ratio': 2.927995235765788, 'n_components': 9}


In [14]:
best_params_pacmap = study_pacmap.best_params

Z_decoupled = executar_pacmap(
    X_data,
    best_params_pacmap
)

score_dr_decoupled, q_local_decoupled, auc_qnn_decoupled = (
    calcular_score_reducao(X_data, Z_decoupled)
)

print(f"Qlocal: {q_local_decoupled:.4f}")
print(f"AUC QNN: {auc_qnn_decoupled:.4f}")
print(f"Score DR: {score_dr_decoupled:.4f}")

Note: `n_components != 2` have not been thoroughly tested.


Qlocal: 0.4967
AUC QNN: 0.7744
Score DR: 0.6355


## Função Objetivo GMM

In [15]:
def objective_gmm(trial):

    params = {
        "n_components": trial.suggest_int(
            "n_components", 3, 7
        ),

        "covariance_type": trial.suggest_categorical(
            "covariance_type",
            ["full", "tied", "diag", "spherical"]
        ),

        "n_init": trial.suggest_int(
            "n_init", 5, 15
        ),

        "random_state": 42
    }

    try:
        gmm = GaussianMixture(**params)

        labels = gmm.fit_predict(Z_decoupled)

        score_gmm, sil_score, ch_score, ch_score_norm = (
            calcular_score_clustering(Z_decoupled, labels)
        )

    except Exception as error:
        print(f"Erro no GMM ou nas métricas: {error}")
        raise optuna.TrialPruned()

    trial.set_user_attr("Silhouette", float(sil_score))
    trial.set_user_attr("Calinski_Harabasz", float(ch_score))
    trial.set_user_attr("CH_normalizado", float(ch_score_norm))

    return score_gmm

## Execução Optuna GMM

In [16]:
study_gmm = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42),
    study_name="GMM_VST_MAD_5000",
    storage=f"sqlite:///{OPTUNA_DIR / 'gmm_vst_mad_5000.db'}",
    load_if_exists=True
)

study_gmm.optimize(
    objective_gmm,
    n_trials=200
)

print("Melhor score:", study_gmm.best_value)

print("\nMelhores hiperparâmetros:")
print(study_gmm.best_params)

Melhor score: 0.6752004259972537

Melhores hiperparâmetros:
{'n_components': 4, 'covariance_type': 'spherical', 'n_init': 13}


In [17]:
best_params_gmm = study_gmm.best_params

gmm_decoupled = GaussianMixture(
    n_components=best_params_gmm["n_components"],
    covariance_type=best_params_gmm["covariance_type"],
    n_init=best_params_gmm["n_init"],
    random_state=42
)

labels_decoupled = gmm_decoupled.fit_predict(Z_decoupled)

score_gmm_decoupled, sil_decoupled, ch_decoupled, ch_norm_decoupled = (
    calcular_score_clustering(Z_decoupled, labels_decoupled)
)

ari_decoupled = adjusted_rand_score(
    y_true,
    labels_decoupled
)

print("Resultados duas etapas")
print("------------------------------")
print(f"Qlocal: {q_local_decoupled:.4f}")
print(f"AUC QNN: {auc_qnn_decoupled:.4f}")
print(f"Silhouette: {sil_decoupled:.4f}")
print(f"Calinski-Harabasz: {ch_decoupled:.4f}")
print(f"ARI: {ari_decoupled:.4f}")

Resultados duas etapas
------------------------------
Qlocal: 0.4967
AUC QNN: 0.7744
Silhouette: 0.5174
Calinski-Harabasz: 4987.2372
ARI: 0.3842


## Salvando os resultados

In [19]:
df_embedding_decoupled = pd.DataFrame(
    Z_decoupled,
    index=df_pca.index,
    columns=[
        f"PaCMAP_{i+1}"
        for i in range(Z_decoupled.shape[1])
    ]
)

df_embedding_decoupled["Cluster"] = labels_decoupled
df_embedding_decoupled["SUBTYPE"] = y_true

df_embedding_decoupled.to_csv(
    EMBEDDINGS_DIR / "embedding_decoupled_vst_mad_5000.csv"
)

joblib.dump(
    gmm_decoupled,
    EMBEDDINGS_DIR / "gmm_decoupled_vst_mad_5000.pkl"
)

with open(
    OPTUNA_DIR / "best_params_gmm_vst_mad_5000.json",
    "w"
) as file:
    json.dump(best_params_gmm, file, indent=4)

study_gmm.trials_dataframe().to_csv(
    OPTUNA_DIR / "trials_gmm_vst_mad_5000.csv",
    index=False
)

In [20]:
# Salvar parâmetros do PaCMAP
with open(
    OPTUNA_DIR / "best_params_pacmap_vst_mad_5000.json",
    "w"
) as file:
    json.dump(best_params_pacmap, file, indent=4)


# Salvar parâmetros do GMM
with open(
    OPTUNA_DIR / "best_params_gmm_vst_mad_5000.json",
    "w"
) as file:
    json.dump(best_params_gmm, file, indent=4)

In [21]:
# ============================================================
# SALVAMENTO DOS RESULTADOS (CENÁRIO DESACOPLADO)
# ============================================================

# 1. Salvar DataFrame do Embedding 2D
df_embedding_decoupled = pd.DataFrame(
    Z_decoupled,
    index=df_pca.index,
    columns=[
        f"PaCMAP_{i+1}"
        for i in range(Z_decoupled.shape[1])
    ]
)

df_embedding_decoupled["Cluster"] = labels_decoupled
df_embedding_decoupled["SUBTYPE"] = y_true

df_embedding_decoupled.to_csv(
    EMBEDDINGS_DIR / "embedding_decoupled_vst_mad_5000.csv"
)

# 2. Salvar Modelo GMM Treinado
joblib.dump(
    gmm_decoupled,
    EMBEDDINGS_DIR / "gmm_decoupled_vst_mad_5000.pkl"
)

# 3. Salvar Histórico dos Ensaios em CSV
study_pacmap.trials_dataframe().to_csv(
    OPTUNA_DIR / "trials_pacmap_vst_mad_5000.csv",
    index=False
)

study_gmm.trials_dataframe().to_csv(
    OPTUNA_DIR / "trials_gmm_vst_mad_5000.csv",
    index=False
)

# 4. OPÇÃO B: Salvar informações completas dos best_trials (Parâmetros + Métricas user_attrs)
best_trial_pacmap = study_pacmap.best_trial
best_trial_gmm = study_gmm.best_trial

best_info_decoupled = {
    "pacmap_optimization": {
        "best_trial_number": best_trial_pacmap.number,
        "best_value": best_trial_pacmap.value,
        "params": best_trial_pacmap.params,
        "user_attrs": best_trial_pacmap.user_attrs
    },
    "gmm_optimization": {
        "best_trial_number": best_trial_gmm.number,
        "best_value": best_trial_gmm.value,
        "params": best_trial_gmm.params,
        "user_attrs": best_trial_gmm.user_attrs
    }
}

with open(
    OPTUNA_DIR / "best_trial_info_decoupled_vst_mad_5000.json",
    "w"
) as file:
    json.dump(best_info_decoupled, file, indent=4)

# 5. Salvar apenas os dicionários de hiperparâmetros (compatibilidade prévia)
with open(
    OPTUNA_DIR / "best_params_pacmap_vst_mad_5000.json",
    "w"
) as file:
    json.dump(best_params_pacmap, file, indent=4)

with open(
    OPTUNA_DIR / "best_params_gmm_vst_mad_5000.json",
    "w"
) as file:
    json.dump(best_params_gmm, file, indent=4)